# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [1]:
import asyncio
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [2]:
DATA_DIR = Path('../data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [3]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    # TODO: return a messages list like [{'role': 'user', 'content': '...'}]
    prompt = f"""Extract following information from this job posting:
    1. company - the name of the hiring company
    2. role - the job title or position name
    3. years_experience_required - minimum years of experience required (or null if not stated)

    Return ONLY a JSON object with these exact field names. Do not include any extra text.
    If a field is not stated in the snippet, return null for that field.

    Job Posting:
    {snippet_text}

    JSON:"""
    
    return [{'role': 'user', 'content': prompt}]


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    prompt = f"""Extract the following information from job postings:
    1. company: the name of the hiring company
    2. role: the job title or position name
    3. years_experience_required: minimum years of experience required (or null if not stated)
    
    Return ONLY a JSON object with these exact field names. If a field is not stated, return null.
    
    Here are 3 examples:
    
    EXAMPLE 1:
    Snippet: "Acme Corp is hiring a Senior Software Engineer. We need someone with 5+ years of Python experience."
    Output: {{"company": "Acme Corp", "role": "Senior Software Engineer", "years_experience_required": 5}}
    
    EXAMPLE 2:
    Snippet: "DataFlow Inc seeks a Data Analyst with 2 years minimum SQL experience. Great benefits!"
    Output: {{"company": "DataFlow Inc", "role": "Data Analyst", "years_experience_required": 2}}
    
    EXAMPLE 3:
    Snippet: "TechHub is looking for a DevOps Engineer. We welcome candidates at any experience level. No minimum years required."
    Output: {{"company": "TechHub", "role": "DevOps Engineer", "years_experience_required": null}}
    
    Now extract from this job posting:
    {snippet_text}
    
    JSON:"""
    
    return [{'role': 'user', 'content': prompt}]


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    system_prompt = """You are an expert HR recruiter tasked with extracting structured information from job postings.

    Your job is to extract exactly three fields from each job posting:
    1. company (string): The exact name of the company hiring. Do not infer or modify.
    2. role (string): The exact job title or position name. Do not infer or modify.
    3. years_experience_required (integer or null): The minimum years of experience required.
       - If the posting states a specific number (e.g., "5 years", "3+"), extract that number as an integer.
       - If the posting says "5+ years", extract 5.
       - If the posting says "around 5 years", extract 5.
       - If the posting does NOT state a minimum years requirement, return null.
       - DO NOT infer, guess, or hallucinate a number if it is not explicitly stated.
       - DO NOT return null unless the posting clearly states no requirement.
    
    You MUST output ONLY a valid JSON object with these exact field names:
    {
      "company": <string or null>,
      "role": <string or null>,
      "years_experience_required": <integer or null>
    }
    
    Do not include any other text, explanation, or markdown formatting."""
    
    user_prompt = f"""Extract information from this job posting:
    
    {snippet_text}"""
    
    return [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    prompt = f"""Extract company, role, and years_experience_required from this job posting.

    IMPORTANT: Think step by step before answering.
    1. First, identify the company name mentioned in the posting.
    2. Then, identify the job title or role.
    3. Finally, look for any mention of years of experience required:
       - If stated explicitly (e.g., "5 years", "3+"), note that number.
       - If NOT stated, note that it's missing.
    4. Return null for any field that is not stated — do not infer or guess.
    
    After thinking through these steps, output ONLY a JSON object in this format:
    {{"company": <string or null>, "role": <string or null>, "years_experience_required": <integer or null>}}
    
    Job posting:
    {snippet_text}
    
    Let's think step by step:"""
    
    return [{'role': 'user', 'content': prompt}]


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [4]:
import re

def parse_response(text: str) -> dict | None:
    """Extract JSON from response, handling markdown fences and reasoning text."""

    if not text:
        return None

    try:
        # Remove markdown fences (```json ... ```)
        text = text.replace('```json', '').replace('```', '').strip()

        # Find JSON object in the text (handles CoT with reasoning before JSON)
        # Pattern: find { ... } allowing for nested structures
        json_match = re.search(r'\{[^{}]*(?:"[^"]*"[^{}]*)*\}', text, re.DOTALL)

        if json_match:
            json_str = json_match.group()
            result = json.loads(json_str)

            # Normalize field names
            normalized = {}
            normalized['company'] = result.get('company') or result.get('company_name')
            normalized['role'] = result.get('role') or result.get('position') or result.get('job_title')
            normalized['years_experience_required'] = result.get('years_experience_required') or result.get('years')

            return normalized

    except (json.JSONDecodeError, AttributeError):
        return None

    return None

async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return results with cost & latency."""

    start_time = time.time()
    snippet_id = snippet['id']
    snippet_text = snippet['snippet']

    try:
        # Get the strategy function
        strategy_func = STRATEGIES[strategy_name]
        messages = strategy_func(snippet_text)

        # Call the API
        response = await client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=TEMPERATURE,
            max_tokens=200
        )

        # Calculate latency and cost
        latency = time.time() - start_time
        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens
        cost = (input_tokens * RATES[MODEL]['in'] +
                output_tokens * RATES[MODEL]['out'])

        # Extract the text response
        response_text = response.choices[0].message.content

        # Try to parse JSON
        extracted = parse_response(response_text)

        return {
            'strategy': strategy_name,
            'snippet_id': snippet_id,
            'response_text': response_text,
            'extracted': extracted,
            'input_tokens': input_tokens,
            'output_tokens': output_tokens,
            'cost_usd': cost,
            'latency_s': latency,
            'parse_success': extracted is not None,
        }

    except Exception as e:
        latency = time.time() - start_time
        return {
            'strategy': strategy_name,
            'snippet_id': snippet_id,
            'response_text': None,
            'extracted': None,
            'input_tokens': 0,
            'output_tokens': 0,
            'cost_usd': 0,
            'latency_s': latency,
            'parse_success': False,
            'error': str(e),
        }


async def run_all() -> list[dict]:
    """Run all 10 snippets × 4 strategies = 40 calls in parallel."""

    tasks = []

    # Create tasks for all 10 × 4 = 40 combinations
    for snippet in snippets:
        for strategy_name in STRATEGIES.keys():
            task = run_one(strategy_name, snippet)
            tasks.append(task)

    # Run all tasks in parallel
    print(f"Running {len(tasks)} API calls in parallel...")
    results = await asyncio.gather(*tasks)

    print(f"✓ Completed {len(results)} calls")
    return results


In [5]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')
results[0]

Running 40 API calls in parallel...


✓ Completed 40 calls
Got 40 results.


{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'response_text': '{\n    "company": "Acme Corp",\n    "role": "Senior Software Engineer",\n    "years_experience_required": 5\n}',
 'extracted': {'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': 5},
 'input_tokens': 142,
 'output_tokens': 30,
 'cost_usd': 3.93e-05,
 'latency_s': 1.351724624633789,
 'parse_success': True}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [6]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """
    Compare 3 fields. Return 0,1,2, or 3.

    Rules:
      - Strings: case-insensitive, whitespace-trimmed for strings.
      - Years: treat "5+" and 5 as equivalent, null matches null
      - if extraction failed (None), score is 0
    """
    if extracted is None:
      return 0

    score = 0

    # 1. compare company (case-insensitive and trimmed)
    ext_company = extracted.get('company')
    gold_company = gold.get('company')

    if ext_company and gold_company:
      if ext_company.strip().lower() == gold_company.strip().lower():
        score += 1
    elif ext_company is None and gold_company is None:
      score += 1

    #2. Compare role
    ext_role = extracted.get('role')
    gold_role = gold.get('role')
    if ext_role and gold_role:
      if ext_role.strip().lower() == gold_role.strip().lower():
        score += 1
    elif ext_role is None and gold_role is None:
      score += 1

    #3. Compare years (handle "5+" -> 5 , null -> null)
    ext_years = extracted.get('years_experience_required')
    gold_years = gold.get('years_experience_required')

    # Normalize: strip "+" and convert to int if needed
    if ext_years is not None and isinstance(ext_years, str):
      try:
        ext_years = int(ext_years.rstrip('+'))
      except (ValueError, AttributeError):
        ext_years = None
    
    if ext_years == gold_years:
        score += 1

    return score


async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Use gpt-4o as a judge. Return integer 1-4.
    
    Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    """
    try:
        judge_prompt = f"""You are a strict evaluator. Score this extraction on a scale of 1-4.

        Job Posting:
        {snippet_text}

        Expected (Golden):
        {json.dumps(gold, indent=2)}

        Extracted:
        {json.dumps(extracted, indent=2)}

        Scoring Rubric:
        4 — All three fields (company, role, years) are correct
        3 — Two of three fields are correct, no fabricated data
        2 — One of three fields is correct, or the model fabricated a field that wasn't in the posting
        1 — Zero fields correct, or the response was unparsable

        Respond with ONLY a single integer: 1, 2, 3, or 4"""

        response = await client.chat.completions.create(
                    model=JUDGE_MODEL,
                    messages=[{'role': 'user', 'content': judge_prompt}],
                    temperature=0.0,  # Critical: reproducible scoring
                    max_tokens=10
                )

        # Extract the score (should be 1, 2, 3, or 4)
        score_text = response.choices[0].message.content.strip()
        score = int(score_text)

        return max(1, min(4, score))  # Clamp to 1-4 range
    except Exception as e:
      print(f"Judge error: {e}")
      return 2  # Default to 2 on error

In [7]:
# Apply scoring to all 40 results
# TODO: loop through results, attach accuracy + parse_success + llm_judge_score to each row
scored = []   # list of result dicts with scoring fields added

for result in results:
    snippet_id = result['snippet_id']
    gold = golden[snippet_id]
    extracted = result['extracted']
    snippet_text = snippets[[s['id'] for s in snippets].index(snippet_id)]['snippet']

    # Accuracy score (0-3)
    accuracy = score_accuracy(extracted, gold)

    # Add to result
    result['accuracy'] = accuracy
    result['parse_success'] = extracted is not None

    scored.append(result)
print(f'Scored {len(scored)} results.')

Scored 40 results.


## Step 5 — Build the comparison table

In [15]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'cost_usd': 'sum',
    'latency_s': 'median',
}).round(3)

summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Total cost ($)', 'Latency p50 (s)']
summary

# Save to markdown file
comparison_md = summary.to_markdown()

with open('mp1_comparison.md', 'w') as f:
    f.write("# MP1 Comparison Table\n\n")
    f.write(comparison_md)
    f.write("\n\n## Notes\n")
    f.write("- Accuracy: 0-3 scale (count of correct fields)\n")
    f.write("- Cost: Total USD for all 10 snippets\n")
    f.write("- Latency: Median time per call (seconds)\n")
    f.write("- Parse Success: % of responses that parsed as valid JSON\n")

print("✓ Saved to mp1_comparison.md")

✓ Saved to mp1_comparison.md


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```